# Data Profiling

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

orders = pd.read_csv("./olist_orders_dataset.csv")
items = pd.read_csv("./olist_order_items_dataset.csv")
products = pd.read_csv("./olist_products_dataset.csv")
customers = pd.read_csv("./olist_customers_dataset.csv")
payments = pd.read_csv("./olist_order_payments_dataset.csv")
reviews = pd.read_csv("./olist_order_reviews_dataset.csv")
sellers = pd.read_csv("./olist_sellers_dataset.csv")
geoloc = pd.read_csv("./olist_geolocation_dataset.csv")

### Orders' Profile

In [2]:
print('Orders DF Shape')
print(orders.shape)

print('\nOrders DF info')
print(orders.info())

Orders DF Shape
(99441, 8)

Orders DF info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB
None


In [3]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
orders_nulls = orders.isna().sum()
order_null_pct = (orders.isna().mean() * 100).sort_values(ascending=False)
order_dup = orders.duplicated().sum()
order_statuses = orders['order_status'].value_counts()

print('Null Count')
print(orders_nulls)

print('\nNull Percentage')
print(order_null_pct)     

print('\nDuplicated Records Count')
print(order_dup)

print('\nStatuses Count')
print(order_statuses)

Null Count
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Null Percentage
order_delivered_customer_date    2.981668
order_delivered_carrier_date     1.793023
order_approved_at                0.160899
order_id                         0.000000
order_purchase_timestamp         0.000000
order_status                     0.000000
customer_id                      0.000000
order_estimated_delivery_date    0.000000
dtype: float64

Duplicated Records Count
0

Statuses Count
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [5]:
# Evaluate if we have unique keys in both orders and customer ids
orders['order_id'].nunique(), len(orders), orders['customer_id'].nunique()

(99441, 99441, 99441)

In [6]:
# Orders' dates transformation to datetime datatype

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_cols] = orders[date_cols].apply(pd.to_datetime)

print(orders[date_cols].dtypes)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [7]:
# Check dates consistency - Check --> purchase <= approved <= delivered_carrier <= delivered_customer

valid_rows = orders[
    orders[['order_purchase_timestamp',
            'order_approved_at',
            'order_delivered_carrier_date',
            'order_delivered_customer_date']].notna().all(axis=1)
    ]

inconsistent_dates  = valid_rows[
    (orders['order_purchase_timestamp'] > orders['order_approved_at']) |
    (orders['order_approved_at'] > orders['order_delivered_carrier_date']) |
    (orders['order_delivered_carrier_date'] > orders['order_delivered_customer_date'])
                             ]
print('Inconsistent Dates Records')
print(len(inconsistent_dates))

Inconsistent Dates Records
1373


C:\Users\JuanJoseOrtizPescado\AppData\Local\Temp\ipykernel_10820\389572307.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  inconsistent_dates  = valid_rows[


### Order Items

In [8]:
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [9]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [10]:
# Check negative or zero prices/freight
negative_price_items = (items['price'] <= 0).sum()
negative_freight_items = (items['freight_value'] <= 0).sum()

print('Records with negative or zero prices')
print(negative_price_items)

print('Records with negative or zero freight')
print(negative_freight_items)

Records with negative or zero prices
0
Records with negative or zero freight
383


In [11]:
duplicated_order_items = items.duplicated(subset=['order_id', 'order_item_id']).sum()

print('Duplicated order-item records')
print(duplicated_order_items)

Duplicated order-item records
0


In [12]:
items_per_order = items.groupby('order_id')['order_item_id'].count()

print('Distribution of number of items per order')
print(items_per_order.value_counts().sort_index())

Distribution of number of items per order
order_item_id
1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
20        2
21        1
Name: count, dtype: int64


In [13]:
# Change Date from objecto to datetime data type
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])
items['shipping_limit_date'].dtype

dtype('<M8[ns]')

### Products

In [14]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [15]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [16]:
print('Duplicated Product ids count')
print(products['product_id'].duplicated().sum())

print('Number of records with null category')
print(products['product_category_name'].isna().sum(),'-', products['product_category_name'].isna().mean().round(3) * 100,'%')


print('Records with invalid product values')
invalid_product_values = products[
    (products['product_weight_g'] <= 0) |
    (products['product_length_cm'] <= 0) |
    (products['product_height_cm'] <= 0) |
    (products['product_width_cm'] <= 0)
]
print(len(invalid_product_values))

Duplicated Product ids count
0
Number of records with null category
610 - 1.9 %
Records with invalid product values
4


### Customers

In [17]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [18]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [19]:
print("Duplicated customer_id")
print(customers['customer_id'].duplicated().sum())

print("\nUnique customers")
print("customer_id:", customers['customer_id'].nunique())
print("customer_unique_id:", customers['customer_unique_id'].nunique())

print("\nCustomers per unique id")
print(
    customers.groupby('customer_unique_id')['customer_id']
    .nunique()
    .value_counts()
    .sort_index()
)

print("\nStates distribution")
print(customers['customer_state'].value_counts())

print("\nCities count")
print(customers['customer_city'].nunique())

print("\nZip code prefix length")
print(customers['customer_zip_code_prefix'].astype(str).str.len().value_counts())

Duplicated customer_id
0

Unique customers
customer_id: 99441
customer_unique_id: 96096

Customers per unique id
customer_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

States distribution
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

Cities count
4119

Zip code prefix length
customer_zip_code_prefix
5    75446
4    23995
Name: count, dtype: int64


### Payments

In [20]:
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [21]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [ ]:
print('Duplicated Order Ids')
print(payments['order_id'].duplicated().sum())

print('\nRecords with multiple payments')
print(payments.groupby('order_id').size().value_counts().sort_index())

print('\nRecords with invalid payment values')
payments_neg_or_null = len(payments[payments['payment_value'] <= 0])
print(payments_neg_or_null)

print('\nPayment Types')
print(payments['payment_type'].value_counts(dropna=False))

print('\nPayments Installments')
print(payments['payment_installments'].value_counts().sort_index())

print('\nInvalid Payments Installments')
print((payments['payment_installments'] <= 0).sum())

Duplicated Order Ids
4446

Records with multiple payments
1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

Records with invalid payment values
9

Payment Types
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Payments Installments
payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64

Invalid Payments Installments
2


### Reviews

In [35]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [36]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [51]:
print('Duplicated Review Id')
print(reviews['review_id'].duplicated().sum())

print('\nReviews per Order')
print(reviews.groupby('order_id')['review_id'].size().value_counts().sort_index())

print('\nReview Score Distribution')
print(reviews['review_score'].value_counts().sort_index())

print("\nNull comments")
print(reviews[['review_comment_title','review_comment_message']].isna().sum())

Duplicated Review Id
814

Orders with Review count
review_id
1    98126
2      543
3        4
Name: count, dtype: int64

Review Score Rank
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Null comments
review_comment_title      87656
review_comment_message    58247
dtype: int64


In [52]:
# Dates consistency
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

(reviews['review_answer_timestamp'] < reviews['review_creation_date']).sum()

np.int64(0)

### Sellers

In [53]:
sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


In [54]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [59]:
print('Duplicated Sellers')
print(sellers['seller_id'].duplicated().sum())

print('\nCities compilation')
print(sellers['seller_city'].value_counts())

print('\nStates compilation')
print(sellers['seller_state'].value_counts())

print('\nZip code prefirx Check')
print(sellers['seller_zip_code_prefix'].astype(str).str.len().value_counts())

Duplicated Sellers
0

Cities compilation
seller_city
sao paulo                                 694
curitiba                                  127
rio de janeiro                             96
belo horizonte                             68
ribeirao preto                             52
                                         ... 
ipua                                        1
muqui                                       1
timoteo                                     1
pouso alegre                                1
rio de janeiro, rio de janeiro, brasil      1
Name: count, Length: 611, dtype: int64

States compilation
seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
MS       5
RN       5
MT       4
RO       2
SE       2
AC       1
PI       1
MA       1
AM       1
PA       1
Name: count, dtype: int64

Zip code prefirx Check
seller_zip_code_prefix
5    2068
4    1027
Name: count, dtype: int

### Geolocation

In [60]:
geoloc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB


In [61]:
geoloc.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [63]:
print('Duplicated Geolocations')
print(geoloc.duplicated().sum())

print('\nZip code prefix Check')
print(sellers['seller_zip_code_prefix'].astype(str).str.len().value_counts())

Duplicated Geolocations
261831
Lat/Long Description
       geolocation_lat  geolocation_lng
count     1.000163e+06     1.000163e+06
mean     -2.117615e+01    -4.639054e+01
std       5.715866e+00     4.269748e+00
min      -3.660537e+01    -1.014668e+02
25%      -2.360355e+01    -4.857317e+01
50%      -2.291938e+01    -4.663788e+01
75%      -1.997962e+01    -4.376771e+01
max       4.506593e+01     1.211054e+02

Zip code prefirx Check
seller_zip_code_prefix
5    2068
4    1027
Name: count, dtype: int64


## Data Quality Summary

### Orders
The dataset has 99,441 records across 8 columns. Key identifiers (`order_id`, `customer_id`,
`order_status`) are clean — no nulls or duplicates. Date columns came in as `object` and needed
casting to `datetime`. The nulls in `order_delivered_customer_date` (2,965) and
`order_delivered_carrier_date` (1,783) are expected, since 2,963 orders never reached
`delivered` status. One thing worth flagging: 1,373 records have timestamps out of sequence
(purchase → approved → carrier → customer), which will need to be handled in cleaning.

### Order Items
112,650 records, 7 columns, no nulls or duplicates on the composite key (`order_id`,
`order_item_id`). The `shipping_limit_date` column needed a cast to `datetime`. No negative
prices, but 383 records have `freight_value` of zero — these could be free shipping promotions
or data gaps. Most orders contain a single item (88,863); orders with more than 3 items are
uncommon.

### Products
32,951 records across 9 columns, no duplicate `product_id` values. 610 records (1.85%) have no
category assigned, which cascades into nulls across the name, description and photo count
columns. Physical dimensions look mostly fine — just 2 residual nulls — but 4 records have
weight or dimension values at or below zero, which is physically impossible. Two column names
also have a typo (`product_name_lenght`, `product_description_lenght`) that needs renaming
before use.

### Customers
99,441 records, 5 columns, completely clean on `customer_id`. There are 96,096 unique
`customer_unique_id` values, meaning 3,345 real customers made more than one purchase under
different `customer_id` entries. The `customer_zip_code_prefix` column was loaded as `int64`,
which silently dropped the leading zero on 23,995 four-digit prefixes — zero-padding will be
required before joining with geolocation.

### Payments
103,886 records, 5 columns, no nulls. The 4,446 duplicate `order_id` values are intentional —
a single order can have multiple payment rows (e.g. credit card + voucher). That said, 9 records
have a `payment_value` at or below zero, 2 have `payment_installments` of zero, and 3 have
`payment_type` set to `not_defined`. All 14 are invalid and should be removed.

### Reviews
99,224 records across 7 columns. The free-text fields are heavily sparse:
`review_comment_title` is null 88.3% of the time and `review_comment_message` 58.7%, which
limits their analytical value. There are 814 duplicate `review_id` values and 547 orders with
more than one review attached. Date consistency is clean — no review answer comes before its
creation date.

### Sellers
3,095 records, 4 columns, no nulls or duplicates on `seller_id`. Same zip code issue as
customers: `seller_zip_code_prefix` loaded as `int64`, producing 1,027 four-digit prefixes
with a missing leading zero. At least one record also has an invalid format in `seller_city`.

### Geolocation
1,000,163 records, 5 columns, no nulls — but 261,831 of those records are exact duplicates
(26.2% of the total), making this the dataset with the worst structural quality issue. On top
of that, latitude and longitude stats show values well outside Brazil (max lat 45.1°, max lng
121.1°), so invalid coordinates will need to be filtered out before any spatial analysis.